In [3]:

from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from scipy.stats import linregress

from lm_polygraph.utils.manager import UEManager


# ---- Configuration labels & helpers -------------------------------------------------------------

METRIC_LABELS: Dict[str, str] = {
    "Comet-wmt22-comet-da": "Comet",
    "XComet-XCOMET-XXL": "XComet-XXL",
    "metricx-metricx-24-hybrid-large-v2p6": "MetricX-Large",
    "AlignScoreInputOutput": "Align Score",
    "Accuracy": "Acc",
    "Rouge_rougeL": "Rouge-L",
}

UE_METHOD_LABELS: Dict[str, str] = {
    "MaximumSequenceProbability": "MSP",
    "Perplexity": "PPL",
    "MeanTokenEntropy": "MTE",
    "MonteCarloSequenceEntropy": "MCSE",
    "MonteCarloNormalizedSequenceEntropy": "MCNSE",
}


def format_dataset_name(raw_name: str) -> str:
    """
    e.g. 'wmt14_fren' -> 'WMT14 Fr-En'
    """
    try:
        prefix, lang_pair = raw_name.split("_", 1)
        prefix = prefix.upper()
        if len(lang_pair) == 4:
            src = lang_pair[:2].capitalize()
            tgt = lang_pair[2:].capitalize()
            lang_fmt = f"{src}-{tgt}"
        else:
            lang_fmt = lang_pair.upper()
        return f"{prefix} {lang_fmt}"
    except Exception:
        return raw_name.upper()


def safe_minmax(x: np.ndarray) -> np.ndarray:
    """
    MinMax scale 1D array; if variance is zero, return zeros.
    """
    x = np.asarray(x).reshape(-1, 1)
    if np.nanmin(x) == np.nanmax(x):
        return np.zeros_like(x, dtype=float).ravel()
    return MinMaxScaler().fit_transform(x).ravel()


# ---- Data loading & preparation ----------------------------------------------------------------

def build_manager_path(
    dataset: str,
    model: str = "llama",
    model_type: str = "base",
    suffix: str = "test_qe_enriched",
) -> Path:
    """
    Reconstructs the UEManager path used in your original code.
    """
    prefix = "" if model_type == "base" else "_instruct"
    if suffix:
        base = "processed_mans"
        fname = f"{model}{prefix}_{dataset}_train_{suffix}.man"
    else:
        base = "mans"
        fname = f"{model}{prefix}_{dataset}_train.man"
    return Path(base) / fname


def load_manager(
    dataset: str,
    model: str = "llama",
    model_type: str = "base",
    suffix: str = "test_qe_enriched",
) -> UEManager:
    path = build_manager_path(dataset, model=model, model_type=model_type, suffix=suffix)
    return UEManager.load(str(path))


def extract_and_prepare_data_train(
    dataset: str,
    methods_dict: Dict[str, str],
    metrics: Iterable[str],
    model: str = "llama",
    model_type: str = "base",
    suffix: str = "test_qe_enriched",
) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray], np.ndarray]:
    """
    Returns:
        train_ue_values: {short_method: values}
        train_metric_values: {metric_name: values}
        train_gen_lengths: np.ndarray of lengths
    """
    manager = load_manager(dataset, model=model, model_type=model_type, suffix=suffix)

    # sequence lengths
    train_sequences = manager.stats["greedy_tokens"]
    train_gen_lengths = np.array([len(seq) for seq in train_sequences], dtype=float)

    # collect NaN mask across all series we’ll use
    nan_idx: List[int] = []

    train_metric_values: Dict[str, np.ndarray] = {}
    for metric in metrics:
        vals = np.asarray(manager.gen_metrics[("sequence", metric)], dtype=float)
        train_metric_values[metric] = vals
        nan_idx.extend(np.flatnonzero(np.isnan(vals)))

    train_ue_values: Dict[str, np.ndarray] = {}
    for full_name, short_name in methods_dict.items():
        vals = np.asarray(manager.estimations[("sequence", full_name)], dtype=float)
        train_ue_values[short_name] = vals
        nan_idx.extend(np.flatnonzero(np.isnan(vals)))

    # unify & drop NaNs
    nan_idx = np.unique(np.asarray(nan_idx, dtype=int))
    keep_mask = np.ones(train_gen_lengths.shape[0], dtype=bool)
    keep_mask[nan_idx] = False

    train_gen_lengths = train_gen_lengths[keep_mask]
    for k in list(train_metric_values.keys()):
        train_metric_values[k] = train_metric_values[k][keep_mask]
    for k in list(train_ue_values.keys()):
        train_ue_values[k] = train_ue_values[k][keep_mask]

    return train_ue_values, train_metric_values, train_gen_lengths


# ---- Plotting ----------------------------------------------------------------------------------

def plot_metric_vs_length(
    gen_lengths: np.ndarray,
    metric_values: np.ndarray,
    metric_name: str,
    dataset_name: str,
    mode: str = "Train",
    save_path: Path | str = "plot.pdf",
    bins: int = 50,
    show: bool = False,
) -> None:
    """
    Plots binned average metric vs. normalized generated length, with a linear fit on normalized data.
    """
    gen_lengths = np.asarray(gen_lengths, dtype=float)
    metric_values = np.asarray(metric_values, dtype=float)

    # Trim length outliers (5th–95th percentiles) to stabilize view
    lo, hi = np.nanpercentile(gen_lengths, [5, 95])
    sel = (gen_lengths > lo) & (gen_lengths < hi)
    gen_lengths, metric_values = gen_lengths[sel], metric_values[sel]

    # Normalize both axes (stable if constant)
    norm_len = safe_minmax(gen_lengths)
    norm_val = safe_minmax(metric_values)

    # Bin by normalized length (uniform bins)
    df = pd.DataFrame({"length": norm_len, "metric": norm_val})
    df["bin"] = pd.cut(df["length"], bins=bins, labels=False, include_lowest=True)
    grouped = df.groupby("bin", dropna=True)["metric"].agg(["mean", "sem"])
    # x positions are bin centers in [0,1]
    bin_edges = np.linspace(0, 1, bins + 1)
    x_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    # some bins can be empty; align by index
    x_vals = x_centers[grouped.index.values]
    y_vals = grouped["mean"].to_numpy()
    y_errs = grouped["sem"].fillna(0.0).to_numpy()

    # Linear regression (normalized)
    lin = LinearRegression().fit(norm_len.reshape(-1, 1), norm_val)
    slope = float(lin.coef_[0])
    x_line = np.linspace(0, 1, 100)
    y_line = lin.predict(x_line.reshape(-1, 1))

    # p-value via scipy (same normalized data)
    _, _, _, p_val, _ = linregress(norm_len, norm_val)

    # Plot (pure matplotlib, single axes)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x_vals, y_vals, label="AVG metric value", color="navy")  # navy line
    ax.fill_between(
        x_vals, y_vals - y_errs, y_vals + y_errs, alpha=0.2, color="navy"
    )  
    ax.plot(
        x_line, y_line, linestyle="--", color="crimson", label="Regression Line"
    )  

    label = METRIC_LABELS.get(metric_name, metric_name)
    pretty_dataset = format_dataset_name(dataset_name)
    ax.set_title(f"{label} vs. Length ({pretty_dataset}, {mode})", fontsize=14)
    ax.set_xlabel("Generated sequence length (normalized)")
    ax.set_ylabel(f"{label} (normalized)")
    ax.legend()
    ax.text(
        0.02,
        0.98,
        f"Slope: {slope:.2f}\n$p$-value: {p_val:.3f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.9, ec="0.7"),
    )

    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    if show:
        plt.show()
    plt.close(fig)


# ---- Orchestration ------------------------------------------------------------------------------

def generate_metric_trend_plots(
    datasets: Iterable[str],
    metrics: Iterable[str],
    models: Iterable[str],
    methods_dict: Dict[str, str] = UE_METHOD_LABELS,
    model_type: str = "base",
    suffix: str = "full_enriched",
    out_root: Path | str = "plots",
    bins: int = 50,
) -> None:
    out_root = Path(out_root)
    for model in models:
        for dataset in datasets:
            ue_vals, metric_vals, gen_lengths = extract_and_prepare_data_train(
                dataset=dataset,
                methods_dict=methods_dict,
                metrics=metrics,
                model=model,
                model_type=model_type,
                suffix=suffix,
            )


            for metric in metrics:
                out_dir = out_root / model / METRIC_LABELS[metric]
                save_path = out_dir / f"{dataset}_{METRIC_LABELS[metric]}_{model}_train.pdf"
                plot_metric_vs_length(
                    gen_lengths=gen_lengths,
                    metric_values=metric_vals[metric],
                    metric_name=metric,
                    dataset_name=dataset,
                    mode="Train",
                    save_path=save_path,
                    bins=bins,
                    show=False,
                )




In [4]:

if __name__ == "__main__":
    DATASETS = [
        "wmt14_deen",
        "wmt14_fren",
        "wmt14_csen",
        "wmt14_ruen",
        "wmt19_ruen",
        "wmt19_fien",
        "wmt19_deen",
        "wmt19_lten",
    ]
    ALL_METRICS = [
        "Comet-wmt22-comet-da",
        "XComet-XCOMET-XXL",
        "metricx-metricx-24-hybrid-large-v2p6",
    ]
    MODELS = ["llama", "gemma", "eurollm"]

    generate_metric_trend_plots(
        datasets=DATASETS,
        metrics=ALL_METRICS,
        models=MODELS,
        methods_dict=UE_METHOD_LABELS,
        model_type="base",
        suffix="full_enriched",
        out_root="plots",  
        bins=50,
    )


Loading NLI model...
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing GreedyProbsCalculator
Initializing EntropyCalculator
Initializing GreedyLMProbsCalculator
Initializing SamplingGenerationCalculator
Initializing BartScoreCalculator
Initializing ModelScoreCalculat

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f89290b5bd0>]


Loading NLI model...
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing 

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f8928d46140>]


...

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f8930c9ece0>]


Loading NLI model...
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing 

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f893090dae0>]


Loading NLI model...
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing 

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f89293338e0>]
